<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/Autonomous_Scientific_Discovery_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autonomous Scientific Discovery Network & AI Scientist Simulator

This notebook implements a multi-layer framework for modeling the process of scientific discovery using Knowledge Graphs, GNNs, and AI Agents.

In [ ]:
# 1. Install/Update libraries
!pip install --upgrade numpy pandas scipy networkx
!pip install -q pyvis plotly sentence-transformers python-louvain

# 2. Restart kernel to apply changes
import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 14.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 101.6 MB/s eta 0:00:00


In [1]:
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

# Initialize Embedding Model
try:
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    print("Embedder loaded successfully.")
except Exception as e:
    print(f"Embedder loading failed: {e}")

print(f"Setup Complete. Using Numpy version: {np.__version__}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder loaded successfully.
Setup Complete. Using Numpy version: 2.5.0


## 1. Multi-Layer Knowledge Graph Construction

We initialize a Directed Graph (`DiGraph`) to represent the complex relationships between scientific entities. Each node will have a `type` attribute corresponding to one of the 10 layers.

In [5]:
class ScientificKnowledgeGraph:
    def __init__(self):
        self.G = nx.MultiDiGraph()
        self.layers = [
            "Paper", "Author", "Concept", "Dataset", "Experiment",
            "Method", "Material", "Institution", "Patent", "Hypothesis"
        ]

    def add_entity(self, entity_id, entity_type, properties=None):
        if entity_type not in self.layers:
            raise ValueError(f"Invalid layer: {entity_type}")
        self.G.add_node(entity_id, type=entity_type, **(properties or {}))

    def add_relationship(self, source, target, rel_type, weight=1.0):
        self.G.add_edge(source, target, relationship=rel_type, weight=weight)

    def get_layer_nodes(self, layer_name):
        return [n for n, d in self.G.nodes(data=True) if d.get('type') == layer_name]

# Initialize the graph
skg = ScientificKnowledgeGraph()

# Synthetic Seed Data for Demonstration
skg.add_entity("P1", "Paper", {"title": "Attention is All You Need", "year": 2017})
skg.add_entity("C1", "Concept", {"name": "Transformer Architecture"})
skg.add_entity("A1", "Author", {"name": "Vaswani et al."})
skg.add_entity("M1", "Method", {"name": "Self-Attention"})

# Establish Relationships
skg.add_relationship("A1", "P1", "authored")
skg.add_relationship("P1", "C1", "discusses")
skg.add_relationship("P1", "M1", "uses")

print(f"Knowledge Graph initialized with {skg.G.number_of_nodes()} nodes and {skg.G.number_of_edges()} edges.")

Knowledge Graph initialized with 4 nodes and 3 edges.


## 2. Network Science & Discovery Metrics

We analyze the graph to find high-influence papers (Centrality) and research silos (Community Detection).

In [1]:
import community as community_louvain
import networkx as nx
import pandas as pd

def analyze_discovery_network(skg_obj):
    # Ensure the graph has nodes
    if not skg_obj.G.nodes:
        return "Graph is empty."

    # Convert MultiDiGraph to Graph for community detection
    G_undirected = nx.Graph(skg_obj.G)

    # Centrality Metrics
    pagerank = nx.pagerank(skg_obj.G, weight='weight')
    betweenness = nx.betweenness_centrality(skg_obj.G)

    # Community Detection (Louvain)
    partition = community_louvain.best_partition(G_undirected)

    metrics_df = pd.DataFrame({
        'Node': list(pagerank.keys()),
        'PageRank': list(pagerank.values()),
        'Betweenness': list(betweenness.values()),
        'Community': [partition[n] for n in pagerank.keys()]
    })

    return metrics_df.sort_values(by='PageRank', ascending=False)

# Run analysis
try:
    discovery_metrics = analyze_discovery_network(skg)
    print("Discovery Metrics (Top Influencers):")
    print(discovery_metrics.head())
except Exception as e:
    print(f"Analysis failed: {e}. Please ensure you have restarted the kernel.")

Analysis failed: name 'skg' is not defined. Please ensure you have restarted the kernel.


## 3. Autonomous Hypothesis Generation

Using link prediction logic and node embeddings, the AI Scientist proposes new relationships between disparate scientific entities.

In [3]:
# Adding more data to ensure hypotheses can be generated
skg.add_entity("C2", "Concept", {"name": "Large Language Models"})
skg.add_entity("C3", "Concept", {"name": "Neural Scaling Laws"})

def generate_hypotheses(skg_obj, target_layer="Concept", threshold=0.5):
    nodes = skg_obj.get_layer_nodes(target_layer)
    if len(nodes) < 2:
        return "Insufficient data for hypothesis generation."

    hypotheses = []
    for i, u in enumerate(nodes):
        for v in nodes[i+1:]:
            if not skg_obj.G.has_edge(u, v):
                u_name = skg_obj.G.nodes[u].get('name', u)
                v_name = skg_obj.G.nodes[v].get('name', v)
                # Simplified similarity for demonstration
                sim = np.random.uniform(0.6, 0.95)

                if sim > threshold:
                    hypotheses.append({
                        "Hypothesis": f"{u_name} may be related to {v_name}",
                        "Confidence": round(sim, 2),
                        "Type": "Predictive Discovery"
                    })

    return pd.DataFrame(hypotheses)

# Generate new hypotheses
new_science = generate_hypotheses(skg)
print("Generated Hypotheses:")
print(new_science)

NameError: name 'skg' is not defined

## 4. Discovery Dashboards & Interactive Visualization

We use `Plotly` for discovery timelines and `Pyvis` for an interactive exploration of the Multi-Layer Scientific Knowledge Graph.

In [4]:
import plotly.graph_objects as go
from pyvis.network import Network
import IPython

def visualize_scientific_graph(skg_obj, filename="discovery_net.html"):
    net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", directed=True)

    # Color mapping for different layers
    color_map = {
        "Paper": "#1f77b4", "Author": "#ff7f0e", "Concept": "#2ca02c",
        "Method": "#d62728", "Material": "#9467bd", "Hypothesis": "#e377c2"
    }

    for n, d in skg_obj.G.nodes(data=True):
        net.add_node(n, label=d.get('name', d.get('title', n)),
                     color=color_map.get(d.get('type'), "#888888"),
                     title=f"Type: {d.get('type')}")

    for u, v, k, d in skg_obj.G.edges(data=True, keys=True):
        net.add_edge(u, v, title=d.get('relationship', 'rel'))

    net.save_graph(filename)
    return filename

# Generate the interactive visualization
vis_path = visualize_scientific_graph(skg)
print(f"Interactive graph saved to {vis_path}. Use Colab's file explorer to download and view.")

NameError: name 'skg' is not defined

## 5. Sankey Diagram: Knowledge Flow

Mapping the progression from Concepts to Methods and finally to Papers.

In [5]:
def plot_knowledge_sankey():
    fig = go.Figure(data=[go.Sankey(
        node = dict(
          pad = 15,
          thickness = 20,
          line = dict(color = "black", width = 0.5),
          label = ["Transformer", "Self-Attention", "Paper: Vaswani", "Hypothesis: LLM Scaling"],
          color = "blue"
        ),
        link = dict(
          source = [0, 1, 0],
          target = [1, 2, 3],
          value = [10, 10, 5]
      ))])

    fig.update_layout(title_text="Scientific Knowledge Flow Diagram", font_size=12)
    fig.show()

plot_knowledge_sankey()